Now that we have the final_dataset, we have to to figure out book text

In [1]:
import os

required_files = [
    "FINAL_DATASET.csv",          # user_id, gutenberg_ids, goodreads_book_keys, ratings
    "gutenberg_fiction_final.csv" # Text#, Title, Authors, LoCC, etc. metadata
]

missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print("Missing files:")
    for f in missing:
        print(f"  - {f}")
    raise FileNotFoundError("Place the missing files in the working directory before continuing.")
else:
    print("All required files found:")
    for f in required_files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  - {f} ({size_mb:.1f} MB)")

All required files found:
  - FINAL_DATASET.csv (10.4 MB)
  - gutenberg_fiction_final.csv (8.3 MB)


In [2]:
import os
import pandas as pd

# Load metadata and the final matched dataset
fiction = pd.read_csv("gutenberg_fiction_final.csv")
final = pd.read_csv("FINAL_DATASET.csv")

# Get unique Gutenberg Text# IDs actually needed
needed_ids = set()
for g in final["gutenberg_ids"].dropna():
    needed_ids.update(int(x) for x in g.split(";"))

print(f"Unique books to download: {len(needed_ids)}")

# Subset metadata to just what we need
to_fetch = fiction[fiction["Text#"].isin(needed_ids)][["Text#", "Title", "Authors"]].copy()
print(f"Metadata rows matched: {len(to_fetch)}")

# Create output folder for downloaded book files
OUTPUT_DIR = "book_texts"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output folder ready: {OUTPUT_DIR}/")

to_fetch.head()

Unique books to download: 1255
Metadata rows matched: 1255
Output folder ready: book_texts/


,Text#,Title,Authors
0,11,Alice's Adventures in Wonderland,"Carroll, Lewis, 1832-1898"
2,15,"Moby-Dick; or, The Whale","Melville, Herman, 1819-1891"
3,16,Peter Pan,"Barrie, J. M. (James Matthew), 1860-1937"
5,24,O Pioneers!,"Cather, Willa, 1873-1947"
6,27,Far from the Madding Crowd,"Hardy, Thomas, 1840-1928"


In [3]:
import pandas as pd, requests, os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

OUTPUT_DIR = "book_texts"
os.makedirs(OUTPUT_DIR, exist_ok=True)
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0 Safari/537.36"}

fiction = pd.read_csv("gutenberg_fiction_final.csv")
final = pd.read_csv("FINAL_DATASET.csv")
needed_ids = set()
for g in final["gutenberg_ids"].dropna():
    needed_ids.update(int(x) for x in g.split(";"))
to_fetch = fiction[fiction["Text#"].isin(needed_ids)][["Text#", "Title", "Authors"]].copy()

session = requests.Session()
session.headers.update(HEADERS)

def gutenberg_urls(tid):
    return [
        f"https://www.gutenberg.org/cache/epub/{tid}/pg{tid}.txt",
        f"https://www.gutenberg.org/files/{tid}/{tid}-0.txt",
        f"https://www.gutenberg.org/files/{tid}/{tid}.txt",
    ]

def fetch_one(tid):
    out_path = os.path.join(OUTPUT_DIR, f"{tid}.txt")
    tmp_path = out_path + ".part"
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        return (tid, "skipped")
    for url in gutenberg_urls(tid):
        try:
            resp = session.get(url, timeout=15)
            if resp.status_code == 200 and len(resp.content) > 1000:
                with open(tmp_path, "wb") as f:
                    f.write(resp.content)
                os.replace(tmp_path, out_path)
                return (tid, "ok")
        except requests.RequestException:
            continue
    return (tid, "failed")

FAILED_LOG = []
ids = to_fetch["Text#"].astype(int).tolist()

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(fetch_one, tid): tid for tid in ids}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading"):
        tid, status = future.result()
        if status == "failed":
            FAILED_LOG.append(tid)

print(f"Failed: {len(FAILED_LOG)}")
print(FAILED_LOG[:20])

Downloading: 100%|██████████| 1255/1255 [01:30<00:00, 13.84it/s] 

Failed: 38
[6539, 6542, 6538, 9147, 9272, 9338, 9345, 9355, 8809, 8204, 9028, 9360, 9420, 9422, 9685, 9437, 9438, 9687, 9732, 9038]


In [4]:
print(FAILED_LOG)
failed_ids = FAILED_LOG

[6539, 6542, 6538, 9147, 9272, 9338, 9345, 9355, 8809, 8204, 9028, 9360, 9420, 9422, 9685, 9437, 9438, 9687, 9732, 9038, 12702, 12704, 12703, 12708, 12710, 20028, 21171, 22454, 22787, 22949, 23077, 26226, 26274, 26289, 28840, 26290, 26298, 8986]


In [7]:
import requests, os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

OUTPUT_DIR = "book_texts"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0 Safari/537.36"}

session = requests.Session()
session.headers.update(HEADERS)
retry_strategy = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry_strategy))

def gutenberg_urls(tid):
    return [
        f"https://www.gutenberg.org/files/{tid}/{tid}-0.txt",
        f"https://www.gutenberg.org/cache/epub/{tid}/pg{tid}.txt",
        f"https://www.gutenberg.org/files/{tid}/{tid}.txt",
    ]

def fetch_one(tid):
    out_path = os.path.join(OUTPUT_DIR, f"{tid}.txt")
    tmp_path = out_path + ".part"
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        return (tid, "skipped")
    for url in gutenberg_urls(tid):
        try:
            resp = session.get(url, timeout=30)
            if resp.status_code == 200 and len(resp.content) > 1000:
                with open(tmp_path, "wb") as f:
                    f.write(resp.content)
                os.replace(tmp_path, out_path)
                return (tid, "ok")
        except requests.RequestException:
            continue
    return (tid, "failed")


STILL_FAILED = []
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(fetch_one, tid): tid for tid in failed_ids}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Retrying"):
        tid, status = future.result()
        if status == "failed":
            STILL_FAILED.append(tid)

print(f"Still failed: {len(STILL_FAILED)}")
print(STILL_FAILED)

Retrying: 100%|██████████| 38/38 [00:38<00:00,  1.00s/it]

Still failed: 38
[6538, 6542, 6539, 9147, 9338, 9272, 9345, 9355, 8809, 8204, 9028, 9360, 9420, 9685, 9422, 9437, 9438, 9687, 9732, 12702, 9038, 12704, 12708, 12703, 12710, 20028, 21171, 22787, 22454, 22949, 23077, 26226, 26274, 26289, 28840, 26290, 26298, 8986]


In [8]:
still_failed = STILL_FAILED

In [9]:
for tid in still_failed:
    r = session.get(f"https://www.gutenberg.org/ebooks/{tid}", timeout=15)
    has_txt = "Plain Text UTF-8" in r.text
    print(tid, "text available" if has_txt else "audio/other only")

6538 audio/other only
6542 audio/other only
6539 audio/other only
9147 audio/other only
9338 audio/other only
9272 audio/other only
9345 audio/other only
9355 audio/other only
8809 audio/other only
8204 audio/other only
9028 audio/other only
9360 audio/other only
9420 audio/other only
9685 audio/other only
9422 audio/other only
9437 audio/other only
9438 audio/other only
9687 audio/other only
9732 audio/other only
12702 audio/other only
9038 audio/other only
12704 audio/other only
12708 audio/other only
12703 audio/other only
12710 audio/other only
20028 audio/other only
21171 audio/other only
22787 audio/other only
22454 audio/other only
22949 audio/other only
23077 audio/other only
26226 audio/other only
26274 audio/other only
26289 audio/other only
28840 audio/other only
26290 audio/other only
26298 audio/other only
8986 audio/other only


In [11]:
final = pd.read_csv("FINAL_DATASET.csv")
removed_ids = set(still_failed)
def filter_row(row):
    ids = str(row["gutenberg_ids"]).split(";")
    keys = str(row["goodreads_book_keys"]).split(";")
    ratings = str(row["ratings"]).split(";")
    keep_idx = [i for i, tid in enumerate(ids) if int(tid) not in removed_ids]
    return pd.Series({
        "gutenberg_ids": ";".join(ids[i] for i in keep_idx),
        "goodreads_book_keys": ";".join(keys[i] for i in keep_idx),
        "ratings": ";".join(ratings[i] for i in keep_idx),
        "num_books_rated": len(keep_idx),
    })

filtered_cols = final.apply(filter_row, axis=1)
final_clean = final[["user_id"]].join(filtered_cols)
final_clean = final_clean[final_clean["num_books_rated"] >= 8].reset_index(drop=True)

final_clean.to_csv("FINAL_DATASET_with_books.csv", index=False)
print(final_clean.shape)

(20352, 5)
